# Notebook 3: Regression Analysis

In Notebook 2 we could visually see that the value premium has weakened over time.
But is that decline statistically real, or just random noise?

In this notebook we use OLS (Ordinary Least Squares) regression to formally test this.
We run two tests:

1. **Time Trend Regression:** Does the spread decline a little bit every month over time?
2. **Post-2007 Dummy Regression:** Did the value premium specifically collapse after the 
   2008 financial crisis?

A result is considered statistically significant if the t-statistic is above 2.0 
or below -2.0. This means we are 95% confident the result is real and not due to chance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

plt.style.use('seaborn-v0_8-whitegrid')

bm_wide = pd.read_csv('bm_wide.csv', index_col='date', parse_dates=True)
ep_wide = pd.read_csv('ep_wide.csv', index_col='date', parse_dates=True)

bm = bm_wide[bm_wide.index >= '1963-01-01'].copy()
ep = ep_wide[ep_wide.index >= '1963-01-01'].copy()

print("Date range:", bm.index.min(), "to", bm.index.max())

Date range: 1963-01-31 00:00:00 to 2024-12-31 00:00:00


### Regression 1: Time Trend Test

We create a "time trend" variable that counts up by 1 each month (1, 2, 3, 4...).
If the coefficient on this variable is negative and statistically significant,
it means the value premium has been declining a little bit every single month.

In [4]:
bm['time_trend'] = np.arange(len(bm))
ep['time_trend'] = np.arange(len(ep))

X_bm = sm.add_constant(bm['time_trend'])
y_bm = bm['spread']
model_bm = sm.OLS(y_bm, X_bm).fit()

X_ep = sm.add_constant(ep['time_trend'])
y_ep = ep['spread']
model_ep = sm.OLS(y_ep, X_ep).fit()

print("=== BM Time Trend Regression ===")
print(model_bm.summary())

print("\n=== EP Time Trend Regression ===")
print(model_ep.summary())

=== BM Time Trend Regression ===
                            OLS Regression Results                            
Dep. Variable:                 spread   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     5.486
Date:                Tue, 05 May 2026   Prob (F-statistic):             0.0194
Time:                        17:53:36   Log-Likelihood:                -1940.1
No. Observations:                 744   AIC:                             3884.
Df Residuals:                     742   BIC:                             3893.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.07

In [8]:
print("=== KEY RESULTS SUMMARY ===\n")

print("BM Time Trend:")
print(f"  Coefficient: {model_bm.params['time_trend']:.6f}")
print(f"  T-statistic: {model_bm.tvalues['time_trend']:.3f}")
print(f"  P-value:     {model_bm.pvalues['time_trend']:.3f}")

print("\nEP Time Trend:")
print(f"  Coefficient: {model_ep.params['time_trend']:.6f}")
print(f"  T-statistic: {model_ep.tvalues['time_trend']:.3f}")
print(f"  P-value:     {model_ep.pvalues['time_trend']:.3f}")


=== KEY RESULTS SUMMARY ===

BM Time Trend:
  Coefficient: -0.001314
  T-statistic: -2.342
  P-value:     0.019

EP Time Trend:
  Coefficient: -0.000333
  T-statistic: -0.659
  P-value:     0.510


### Interpretation: Regression 1 Results

The BM value premium shows a statistically significant decline over time 
(t = -2.342, p = 0.019), meaning we can say with 95% confidence that the 
BM spread has been shrinking month by month since 1963.

The EP value premium does NOT show a statistically significant decline 
(t = -0.659, p = 0.510). This suggests that how you define value matters,
Book-to-Market may have weakened as book value becomes less meaningful for 
modern tech-heavy companies, while Earnings-to-Price remains more stable.

### Regression 2: Post-2007 Dummy Test

Instead of a gradual time trend, maybe the value premium collapsed specifically 
after the 2008 financial crisis. We test this by creating a dummy variable that 
equals 0 before 2007 and 1 after 2007. If the coefficient is negative and 
statistically significant, it means the value premium was meaningfully lower 
in the post-crisis era.

In [7]:
bm['post2007'] = (bm.index >= '2007-01-01').astype(int)
ep['post2007'] = (ep.index >= '2007-01-01').astype(int)

X_bm2 = sm.add_constant(bm['post2007'])
model_bm2 = sm.OLS(bm['spread'], X_bm2).fit()

X_ep2 = sm.add_constant(ep['post2007'])
model_ep2 = sm.OLS(ep['spread'], X_ep2).fit()

print("=== POST-2007 DUMMY RESULTS ===\n")

print("BM Post-2007:")
print(f"  Coefficient: {model_bm2.params['post2007']:.4f}")
print(f"  T-statistic: {model_bm2.tvalues['post2007']:.3f}")
print(f"  P-value:     {model_bm2.pvalues['post2007']:.3f}")

print("\nEP Post-2007:")
print(f"  Coefficient: {model_ep2.params['post2007']:.4f}")
print(f"  T-statistic: {model_ep2.tvalues['post2007']:.3f}")
print(f"  P-value:     {model_ep2.pvalues['post2007']:.3f}")


=== POST-2007 DUMMY RESULTS ===

BM Post-2007:
  Coefficient: -0.8147
  T-statistic: -3.077
  P-value:     0.002

EP Post-2007:
  Coefficient: -0.2242
  T-statistic: -0.938
  P-value:     0.348


### Interpretation: Regression 2 Results

The BM value premium dropped by 0.81% per month after 2007, and this drop is 
highly statistically significant (t = -3.077, p = 0.002). This suggests the 
2008 financial crisis marked a structural break in the BM value premium.

EP again shows no statistically significant change (t = -0.938, p = 0.348),
reinforcing our finding that the definition of value matters. Book-to-Market 
appears to have been particularly affected by the rise of intangible-asset-heavy 
technology companies after 2008, while Earnings-to-Price has been more resilient.